# Deep Learning

# Tutorial 3: Autograd

In this tutorial, we will cover:

- PyTorch Tensor Autograd Functionality: Automatic differentiation

Prerequisites:

- Python, Tensors

Our contacts:

- Niklas Beuter (niklas.beuter@th-luebeck.de)
- Fenja Falta (fenja.falta@th-luebeck.de)

Course:

- Slides and notebooks will be available at https://lernraum.th-luebeck.de/course/view.php?id=5383


In [3]:
# We first import all needed packages
import torch

## Autograd

Autograd in PyTorch is a powerful tool for automatic differentiation, enabling the efficient computation of gradients in neural networks and other computational graphs. Here's a brief overview:

1. **Graph Construction**: During the forward pass, PyTorch builds a [computational graph](https://colah.github.io/posts/2015-08-Backprop/?ref=upstract.com). Nodes represent tensors, while edges correspond to functions (operations) that transform these tensors.

2. **Enable Gradient Tracking**: By setting `requires_grad=True` for a tensor, you tell PyTorch to track all operations on it. This is crucial for gradient computation.

3. **Backward Propagation**: In the backward pass, PyTorch computes gradients by traversing this graph from outputs to inputs. This is done using the chain rule of calculus.

4. **Gradient Calculation**: The gradients are calculated by `torch.autograd.grad` or `.backward()` methods. For $y = f(x)$, PyTorch computes $ \frac{\partial y}{\partial x} $ by backtracking through the graph.


This system allows for efficient and flexible gradient computations, which is essential for training neural networks using gradient-based optimization methods.

See:
- https://pytorch.org/docs/stable/autograd.html
- https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html

### Explanation computational graph

A computational graph tracks all operations done in the equation. An easy example is the following:

![Easy computation graph](https://th.bing.com/th/id/OIP.HVh8qKeaungir9D_3YO6VAHaEu?rs=1&pid=ImgDetMain)

For deep neural networks we need the computational graph e.g. to keep track of all the partial differentations during backpropagation:

<div>
<img src="https://colah.github.io/posts/2015-08-Backprop/img/tree-backprop.png" width="500"/>
</div>

See:
- https://colah.github.io/posts/2015-08-Backprop/?ref=upstract.com

### Autograd with scalar functions

We first make an easy comparison by setting up an own function and its derivative. We compare its outputs with the autograd function of pytorch.

In [6]:
# Torch has "autograd" to automatically compute gradients
# Let's try it out with simple functions first.


# Define the function, x^2+3x+2
def f(x):
    return x**2 + 3 * x + 2


# Manual gradient w.r.t x
def f_grad(x):
    return 2 * x + 3


def autograd(func, x):
    # Initialize an empty list for gradients
    gradients = []

    # Compute the gradient for each element in the tensor
    for xi in x:
        # Compute the function on the i-th element
        y = func(xi)

        # Compute the gradient for the i-th element
        gradients.append(torch.autograd.grad(outputs=y, inputs=xi)[0])

        print(y,xi,gradients)

        # The torch.autograd.grad function is designed to compute gradients of scalar outputs with respect to inputs.
        # In our case, the function f(x) applied to x_tensor results in a vector (a tensor with multiple elements),
        # not a single scalar. Hence, torch.autograd.grad cannot directly compute the gradient for each element
        # of this vector. To resolve this, we loop over each element of x_tensor, treating each function evaluation
        # f(x[i]) as a scalar output, and compute its gradient individually. This way, we are effectively computing
        # the gradient of multiple scalar functions, each dependent on a single element of x_tensor.

    return torch.stack(gradients)


# Compute some function values and gradients
# make sure to set requires_grad=True to enable gradient tracking on the computational graph
x_tensor = torch.arange(-5, 5, 1, dtype=torch.float32, requires_grad=True)
print(x_tensor)

f_value = f(x_tensor)
f_grad = f_grad(x_tensor)
f_autograd = autograd(f, x_tensor)

# We output each value in x_tensor, the original function value from f(x)=x^2+3x+2, the derivative output f_grad(x) and compare it to the output of autograd(x)
# The first output is e.g., x=-5, f(-5)=(-5)^2+3*(-5)+2=12, f_grad(-5)=2*(-5)+3=-7
for i in range(len(x_tensor)):
    print(
        f"x = {x_tensor[i].item():5.2f}: "
        f"f(x) = {f_value[i].item():5.2f}, "
        f"f_grad(x) = {f_grad[i].item():5.2f}, "
        f"autograd(x) = {f_autograd[i].item():5.2f}"
    )

tensor([-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.], requires_grad=True)
tensor(12., grad_fn=<AddBackward0>) tensor(-5., grad_fn=<UnbindBackward0>) [tensor(-7.)]
tensor(6., grad_fn=<AddBackward0>) tensor(-4., grad_fn=<UnbindBackward0>) [tensor(-7.), tensor(-5.)]
tensor(2., grad_fn=<AddBackward0>) tensor(-3., grad_fn=<UnbindBackward0>) [tensor(-7.), tensor(-5.), tensor(-3.)]
tensor(0., grad_fn=<AddBackward0>) tensor(-2., grad_fn=<UnbindBackward0>) [tensor(-7.), tensor(-5.), tensor(-3.), tensor(-1.)]
tensor(0., grad_fn=<AddBackward0>) tensor(-1., grad_fn=<UnbindBackward0>) [tensor(-7.), tensor(-5.), tensor(-3.), tensor(-1.), tensor(1.)]
tensor(2., grad_fn=<AddBackward0>) tensor(0., grad_fn=<UnbindBackward0>) [tensor(-7.), tensor(-5.), tensor(-3.), tensor(-1.), tensor(1.), tensor(3.)]
tensor(6., grad_fn=<AddBackward0>) tensor(1., grad_fn=<UnbindBackward0>) [tensor(-7.), tensor(-5.), tensor(-3.), tensor(-1.), tensor(1.), tensor(3.), tensor(5.)]
tensor(12., grad_fn=<AddBackward0>) ten

### Autograd with tensors

Now, let's create a tensor with requires_grad=True, by which you're telling PyTorch, "Hey, I might need the gradient of this tensor later, so please keep track of all operations performed on it.".

When you perform operations on tensors that have requires_grad=True, PyTorch keeps track of these operations behind the scenes, creating what's known as a computational graph. This graph records the sequence of operations (nodes) applied to tensors (edges), along with the function that produced the output tensor from the input tensors.

In [7]:
x = torch.tensor([2.0], requires_grad=True)

Imagine you have a simple function of tensors $y=x^2$. If $x$ is a tensor with requires_grad=True, after computing $y$ and invoking y.backward(), PyTorch will automatically compute the derivative of $y$ with respect to $x$ and store it in x.grad.

Now, we'll perform a simple operation on the tensor. Let's compute $y=x^2$:

In [8]:
y = x ** 2
print(f"Value of y: {y}")

Value of y: tensor([4.], grad_fn=<PowBackward0>)


To compute the gradient of $y$ with respect to $x$, we call .backward() on $y$. Since $y$ is not a scalar (a single number), if it were a tensor with more elements, you would need to specify the gradient argument in backward(), which is a tensor of matching shape (this will be explained further down). For our case, $y$ is effectively a scalar, so we can directly call backward().

The .backward() method is where the magic happens. Calling it triggers the computation of gradients. It goes through the computational graph in reverse, from the output back to the inputs, applying the chain rule to compute the gradient of the loss function with respect to the weights.

In [9]:
y.backward()

After calling .backward(), the gradient of $y$ with respect to $x$ is stored in x.grad:

In [10]:
print(f"Gradient of y w.r.t x: {x.grad}")

Gradient of y w.r.t x: tensor([4.])


Next, we are going into more complex tensors and calculate the derivatives for them.

In [11]:
# Define two tensors and track computations
t1 = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32, requires_grad=True)

t2 = torch.tensor([[7, 8, 9], [10, 11, 12]], dtype=torch.float32, requires_grad=True)

# Initially, gradients for t1 are None since no operations have been performed
print(f"t1.grad = {t1.grad}")

# grad_fn for t1 is None because it is not a result of an operation
# but directly created from data in the computational graph
print(f"t1.grad_fn = {t1.grad_fn}")

t1.grad = None
t1.grad_fn = None


In [12]:
# Perform element-wise multiplication of t1 and t2
t1_mul_t2 = t1 * t2

# The resulting tensor t1_mul_t2 has grad_fn set to MulBackward0,
# indicating that it's a result of a multiplication operation
print(f"t1_mul_t2 = {t1_mul_t2}")

# Gradients for t1 are still None because backward() has not been called yet
print(f"t1.grad = {t1.grad}")

t1_mul_t2 = tensor([[ 7., 16., 27.],
        [40., 55., 72.]], grad_fn=<MulBackward0>)
t1.grad = None


After `backward()`, `t1.grad` and `t2.grad` are populated.

The gradient of each element of `t1` is equal to the corresponding element in `t2`, and vice versa. This is because the derivative of `t1[i] * t2[i]` w.r.t. `t1[i]` is `t2[i]`, and w.r.t. `t2[i]` is `t1[i]`. (In case you are unsure, please read again about partial derivatives here: [Wikipedia](https://en.wikipedia.org/wiki/Partial_derivative) or [Studyflix](https://studyflix.de/mathematik/partielle-ableitungen-1341))

In [13]:
# Compute gradients of the sum of all elements in t1_mul_t2 with respect to t1 and t2
t1_mul_t2.sum().backward()

# After backward(), t1.grad and t2.grad are populated.
# The gradient at each element in t1 and t2 indicates the rate of change of the sum with respect to that element.
# For element-wise multiplication, the gradient at each element of t1 is equal to the corresponding element
# in t2 and vice versa. This is because the derivative of t1[i] * t2[i] w.r.t. t1[i] is t2[i],
# and w.r.t. t2[i] is t1[i].

print(f"t1.grad = {t1.grad}")
print(f"t2.grad = {t2.grad}")

t1.grad = tensor([[ 7.,  8.,  9.],
        [10., 11., 12.]])
t2.grad = tensor([[1., 2., 3.],
        [4., 5., 6.]])


### Why `.sum()` is needed for `.backward()`:


- PyTorch's `.backward()` function computes gradients with respect to a scalar value. This is essential because gradients are conceptually the rate of change of a scalar value with respect to other variables. If you have a tensor with more than one element and wish to compute gradients with respect to its elements, you need to first reduce it to a scalar. When performing operations between tensors, like `t1 * t2`, the result is another tensor. To compute gradients with respect to the original tensors (`t1` and `t2`), a scalar value is needed for differentiation. The `.sum()` method achieves this by combining all elements of the resulting tensor into a single scalar.

- When `.backward()` is called on the scalar result of `t1_mul_t2.sum()`, it activates the chain rule in reverse throughout the computational graph. It calculates the gradient of the scalar with respect to each element in the tensors involved in the computation (`t1` and `t2`), effectively propagating the gradients backwards.
- The gradients computed in this manner indicate how much each element of `t1` and `t2` would need to change to increase the scalar sum. This approach is frequently utilized in optimization problems, where the scalar often represents a loss function.

**In summary**, `.sum()` is employed to convert the tensor resulting from `t1 * t2` into a scalar, enabling `.backward()` to compute gradients. This procedure is standard in many deep learning applications, particularly in the computation of loss functions, where errors are backpropagated from a single scalar value (the loss) to update model parameters.

In [14]:
# Analyzing the gradient at t2[0,1]. If t2.grad[0,1] is 2, it means that a unit change in t2[0,1] results in a
# change of 2 in the sum. Therefore, increasing t2[0,1] by 3 should increase the sum by 3 * t2.grad[0,1], under
# linear approximation.

# Create a new tensor and add 3 to t2[0,1]
t2_modified = t2.clone()
t2_modified[0, 1] = t2[0, 1] + 3

# Perform the computation again with the modified t2
t1_mul_t2_updated = t1 * t2_modified
updated_sum = t1_mul_t2_updated.sum()

# Compare the change in sum
change_in_sum = updated_sum - t1_mul_t2.sum()
print(f"Change in sum: {change_in_sum}")

Change in sum: 6.0


> ## Exercise
>
> Change the sum about the value $34$ by adapting the values in either `t1` or `t2`.

In [ ]:
# ✏️ TODO adapt t1 or t2


# ✏️ TODO Perform the computation again with the modified t1 or t2. Name the
# t1_mul_t2_updated =

# Perform the sum computation again with the modified t1 or t2
updated_sum = t1_mul_t2_updated.sum()

# Compare the change in sum
change_in_sum = updated_sum - t1_mul_t2.sum()
print(f"Change in sum: {change_in_sum}")

### Does Autograd Work Without the Sum?
Yes, autograd can work without the .sum(), but with a caveat. If the operation you perform results in a tensor rather than a scalar, you must provide a gradient argument to .backward() that matches the shape of the tensor. This gradient argument is essentially a tensor of the same shape that tells PyTorch how to weigh the importance of each element in your non-scalar tensor when computing the gradient.

For instance, if your operation results in a vector (a tensor with a single dimension), and you still want to call .backward(), you would do something like this:

In [15]:
vector_output = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
vector_output.backward(torch.tensor([1.0, 1.0, 1.0]))  # Gradient argument

Here, the gradient argument [1.0, 1.0, 1.0] is a tensor of ones that indicates each element in the vector contributes equally to the gradient computation.

### Practical Implications
**Scalar Output**: In practice, especially in training neural networks, we deal with scalar loss values representing the aggregate error or loss over a batch of data points. Hence, operations like .sum() or .mean() are commonly used to aggregate outputs into a scalar.

**Non-scalar Outputs**: If your computation does not naturally reduce to a scalar (e.g., you're working with individual elements or vectors), and you still want to compute gradients with respect to certain inputs, you must use the gradient argument to specify how the backpropagation should be handled for each element of the output tensor.

In summary, while autograd can technically work without calling .sum(), doing so (or performing a similar aggregation to obtain a scalar) simplifies the process of computing gradients for the majority of use cases in machine learning, where you're typically optimizing a single loss value.

# References

This notebook is adapted from or uses following sources:
Markus Enzweiler, markus.enzweiler@hs-esslingen.de, https://github.com/menzHSE/cv-ml-lecture-notebooks.git